### **Applied Machine Learning Project — GNN Citation Network**

In [ ]:
pip install torch torch_geometric --quiet --break-system-packages 2>/dev/null | tail -3; python3 -c "import torch; print('torch:', torch.__version__)" 2>/dev/null || echo "torch not available"

In [ ]:
!python3 -c "import torch_geometric; print('pyg:', torch_geometric.__version__)" 2>/dev/null || echo "pyg not installed"

In [ ]:
!pip install -q gdown

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1OB1RVCDJSjUeYu34w0ZqBEk-8XE8IxsX

In [ ]:
base_path = '/kaggle/working/project_data'

In [ ]:
import pickle

with open(f'{base_path}/feature.pkl', 'rb') as f:
    feature_file = pickle.load(f)

print(type(feature_file))

In [ ]:
def read_txt(path):
    with open(path, 'r') as f:
        return [list(map(int, line.strip().split())) for line in f]

cite_file = read_txt(f'{base_path}/paper_file_ann.txt')
train_ref_file = read_txt(f'{base_path}/bipartite_train_ann.txt')
test_ref_file = read_txt(f'{base_path}/bipartite_test_ann.txt')
coauthor_file = read_txt(f'{base_path}/author_file_ann.txt')

In [ ]:
print(feature_file.shape)
print(feature_file[:5])

In [ ]:
print(cite_file[:5])
print(train_ref_file[:5])
print(test_ref_file[:5])
print(coauthor_file[:5])

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize, StandardScaler, MinMaxScaler
from collections import defaultdict

# ─────────────────────────────────────────────
# 0. CONFIG — change these paths to your local copies
# ─────────────────────────────────────────────

DATA_DIR = "./project_data"          # folder where you put the Drive files

PAPER_FILE   = os.path.join(DATA_DIR, "paper_file_ann.txt")
AUTHOR_FILE  = os.path.join(DATA_DIR, "author_file_ann.txt")
TRAIN_FILE   = os.path.join(DATA_DIR, "bipartite_train_ann.txt")
TEST_FILE    = os.path.join(DATA_DIR, "bipartite_test_ann.txt")
FEATURE_FILE = os.path.join(DATA_DIR, "feature.pkl")

OUT_DIR = "./cleaned_data"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
PAPER_FILE

# STEP A — Load & Explore Raw Files


In [ ]:
def load_txt_file(path, sep="\t"):
    """
    Generic loader for the .txt annotation files.
    Tries tab-separated first, falls back to space-separated.
    Returns a raw DataFrame and prints basic info.
    """
    print(f"\n{'='*60}")
    print(f"Loading: {os.path.basename(path)}")
    print(f"{'='*60}")

    try:
        df = pd.read_csv(path, sep=sep, header=None, low_memory=False)
    except Exception:
        df = pd.read_csv(path, sep=" ", header=None, low_memory=False)

    print(f"  Shape        : {df.shape}")
    print(f"  Columns      : {list(df.columns)}")
    print(f"  Null counts  :\n{df.isnull().sum()}")
    print(f"\n  First 3 rows:\n{df.head(3)}")
    return df


def load_features(path):
    """Load the pre-extracted feature pickle."""
    print(f"\n{'='*60}")
    print(f"Loading features: {os.path.basename(path)}")
    print(f"{'='*60}")

    with open(path, "rb") as f:
        features = pickle.load(f)

    print(f"  Type   : {type(features)}")
    if hasattr(features, "shape"):
        print(f"  Shape  : {features.shape}")
    elif isinstance(features, dict):
        print(f"  Keys   : {len(features)} entries")
        sample_key = next(iter(features))
        print(f"  Sample : key={sample_key}, value shape={np.array(features[sample_key]).shape}")
    return features

# STEP B — Format & Name Columns

In [ ]:
def format_paper_df(df_raw):
    """
    paper_file_ann.txt — expected columns:
      paper_id | title | abstract | venue | year  (adjust if different)
    """
    n = df_raw.shape[1]
    if n >= 5:
        df_raw.columns = ["paper_id", "title", "abstract", "venue", "year"] + \
                         [f"extra_{i}" for i in range(n - 5)]
    elif n == 4:
        df_raw.columns = ["paper_id", "title", "abstract", "venue"]
    elif n == 3:
        df_raw.columns = ["paper_id", "title", "abstract"]
    else:
        df_raw.columns = [f"col_{i}" for i in range(n)]

    df_raw["paper_id"] = df_raw["paper_id"].astype(str).str.strip()
    return df_raw


def format_author_df(df_raw):
    """
    author_file_ann.txt — expected columns:
      author_id | name | affiliation  (adjust if different)
    """
    n = df_raw.shape[1]
    if n >= 3:
        df_raw.columns = ["author_id", "name", "affiliation"] + \
                         [f"extra_{i}" for i in range(n - 3)]
    elif n == 2:
        df_raw.columns = ["author_id", "name"]
    else:
        df_raw.columns = [f"col_{i}" for i in range(n)]

    df_raw["author_id"] = df_raw["author_id"].astype(str).str.strip()
    return df_raw


def format_edge_df(df_raw, name="edges"):
    """
    bipartite_*_ann.txt — expected columns:
      author_id | paper_id | (optional: label / weight)
    """
    n = df_raw.shape[1]
    if n >= 3:
        df_raw.columns = ["author_id", "paper_id", "label"] + \
                         [f"extra_{i}" for i in range(n - 3)]
    else:
        df_raw.columns = ["author_id", "paper_id"]

    df_raw["author_id"] = df_raw["author_id"].astype(str).str.strip()
    df_raw["paper_id"]  = df_raw["paper_id"].astype(str).str.strip()

    print(f"\n{name} — label distribution:")
    if "label" in df_raw.columns:
        print(df_raw["label"].value_counts())
    return df_raw

#STEP C — Clean Data

In [ ]:
def remove_duplicates(df, subset, name=""):
    before = len(df)
    df = df.drop_duplicates(subset=subset)
    after = len(df)
    print(f"  [{name}] Removed {before - after} duplicates → {after} rows remain")
    return df


def remove_nulls(df, key_cols, name=""):
    before = len(df)
    df = df.dropna(subset=key_cols)
    # Also drop rows where key columns are empty strings
    for col in key_cols:
        if df[col].dtype == object:
            df = df[df[col].str.strip() != ""]
    after = len(df)
    print(f"  [{name}] Removed {before - after} null/empty rows → {after} rows remain")
    return df


def clean_papers(df):
    print("\n── Cleaning papers ──")
    df = remove_nulls(df, ["paper_id"], name="papers")
    df = remove_duplicates(df, subset=["paper_id"], name="papers")
    df = df.reset_index(drop=True)
    return df


def clean_authors(df):
    print("\n── Cleaning authors ──")
    df = remove_nulls(df, ["author_id"], name="authors")
    df = remove_duplicates(df, subset=["author_id"], name="authors")
    df = df.reset_index(drop=True)
    return df


def clean_edges(df, author_ids, paper_ids, name="edges"):
    """Remove edges whose nodes don't exist in the node files."""
    print(f"\n── Cleaning {name} ──")
    before = len(df)
    df = df[df["author_id"].isin(author_ids) & df["paper_id"].isin(paper_ids)]
    after = len(df)
    print(f"  [{name}] Removed {before - after} orphan-node edges → {after} rows remain")
    df = remove_duplicates(df, subset=["author_id", "paper_id"], name=name)
    df = df.reset_index(drop=True)
    return df

# STEP D — Train / Test Split Check

In [ ]:
def check_split(train_df, test_df):
    print("\n── Train / Test Split Check ──")

    train_edges = set(zip(train_df["author_id"], train_df["paper_id"]))
    test_edges  = set(zip(test_df["author_id"],  test_df["paper_id"]))
    overlap     = train_edges & test_edges

    print(f"  Train edges : {len(train_edges)}")
    print(f"  Test  edges : {len(test_edges)}")
    print(f"  Overlap     : {len(overlap)}  {'✓ OK' if len(overlap) == 0 else '⚠ OVERLAP FOUND!'}")

    if "label" in train_df.columns and "label" in test_df.columns:
        print(f"\n  Train label dist:\n{train_df['label'].value_counts()}")
        print(f"\n  Test  label dist:\n{test_df['label'].value_counts()}")

    return overlap

# STEP E — Data Consistency Checks

In [ ]:
def consistency_check(train_df, test_df, authors_df, papers_df, features):
    print("\n── Consistency Checks ──")

    author_ids = set(authors_df["author_id"])
    paper_ids  = set(papers_df["paper_id"])

    # All edge node IDs must be in node files
    for split_name, df in [("train", train_df), ("test", test_df)]:
        missing_authors = set(df["author_id"]) - author_ids
        missing_papers  = set(df["paper_id"])  - paper_ids
        print(f"  [{split_name}] Missing author IDs in edges: {len(missing_authors)}")
        print(f"  [{split_name}] Missing paper  IDs in edges: {len(missing_papers)}")

    # Feature alignment
    total_nodes = len(author_ids) + len(paper_ids)
    if hasattr(features, "shape"):
        n_feat = features.shape[0]
        aligned = (n_feat == total_nodes)
        print(f"\n  Total nodes     : {total_nodes}")
        print(f"  Feature rows    : {n_feat}")
        print(f"  Aligned         : {'✓ OK' if aligned else f'⚠ MISMATCH ({n_feat} vs {total_nodes})'}")
    elif isinstance(features, dict):
        feat_ids = set(str(k) for k in features.keys())
        node_ids = author_ids | paper_ids
        missing_feat = node_ids - feat_ids
        print(f"\n  Nodes missing features : {len(missing_feat)}")

# STEP F — Normalize Features


In [ ]:
def normalize_features(features, method="l2"):
    """
    method: 'l2' | 'minmax' | 'zscore'
    Returns normalized numpy array.
    """
    print(f"\n── Normalizing Features (method={method}) ──")

    import scipy.sparse as sp

    # Convert to dense numpy array if sparse
    if sp.issparse(features):
        print("  Converting sparse matrix to dense...")
        X = features.toarray().astype(np.float32)
    elif isinstance(features, dict):
        # Sort by key and stack
        sorted_keys = sorted(features.keys())
        X = np.array([features[k] for k in sorted_keys], dtype=np.float32)
    else:
        X = np.array(features, dtype=np.float32)

    print(f"  Feature matrix shape : {X.shape}")
    print(f"  Min / Max (before)   : {X.min():.4f} / {X.max():.4f}")

    if method == "l2":
        X_norm = normalize(X, norm="l2")
    elif method == "minmax":
        scaler = MinMaxScaler()
        X_norm = scaler.fit_transform(X)
    elif method == "zscore":
        scaler = StandardScaler()
        X_norm = scaler.fit_transform(X)
    else:
        raise ValueError(f"Unknown method: {method}")

    print(f"  Min / Max (after)    : {X_norm.min():.4f} / {X_norm.max():.4f}")
    return X_norm


# STEP G —  Alternative Feature Types


In [ ]:
def tfidf_features_from_papers(papers_df, text_col="abstract", max_features=512):
    """
    Build TF-IDF features from paper text (title or abstract).
    Returns a numpy array of shape (n_papers, max_features).
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    print(f"\n── Building TF-IDF features from '{text_col}' ──")

    if text_col not in papers_df.columns:
        print(f"  Column '{text_col}' not found, skipping.")
        return None

    texts = papers_df[text_col].fillna("").tolist()
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(texts)

    print(f"  TF-IDF shape : {tfidf_matrix.shape}")
    return tfidf_matrix


def coauthorship_features(train_df, authors_df):
    """
    Build simple co-authorship frequency feature per author:
    how many papers each author has in training set.
    Returns a dict {author_id: paper_count}.
    """
    print("\n── Building Co-authorship Frequency Features ──")
    counts = train_df.groupby("author_id")["paper_id"].count().to_dict()
    # Fill 0 for authors not in training edges
    for aid in authors_df["author_id"]:
        if aid not in counts:
            counts[aid] = 0
    print(f"  Author paper-count range: {min(counts.values())} – {max(counts.values())}")
    return counts

# STEP H — Save Cleaned Files

In [ ]:
def save_all(papers_df, authors_df, train_df, test_df, features_norm, out_dir):
    print(f"\n── Saving to {out_dir} ──")

    papers_df.to_csv(os.path.join(out_dir, "cleaned_papers.csv"),  index=False)
    authors_df.to_csv(os.path.join(out_dir, "cleaned_authors.csv"), index=False)
    train_df.to_csv(os.path.join(out_dir, "train_edges.csv"),  index=False)
    test_df.to_csv(os.path.join(out_dir, "test_edges.csv"),    index=False)

    with open(os.path.join(out_dir, "features_normalized.pkl"), "wb") as f:
        pickle.dump(features_norm, f)

    print("  ✓ cleaned_papers.csv")
    print("  ✓ cleaned_authors.csv")
    print("  ✓ train_edges.csv")
    print("  ✓ test_edges.csv")
    print("  ✓ features_normalized.pkl")


def print_summary(papers_df, authors_df, train_df, test_df, features_norm):
    print("\n" + "="*60)
    print("  FINAL DATA SUMMARY")
    print("="*60)
    print(f"  Papers      : {len(papers_df)}")
    print(f"  Authors     : {len(authors_df)}")
    print(f"  Train edges : {len(train_df)}")
    print(f"  Test edges  : {len(test_df)}")
    if features_norm is not None:
        print(f"  Feature dim : {features_norm.shape}")
    print("="*60)

# MAIN — Run Full Pipeline

In [ ]:
import pandas as pd
import os

def load_txt_file_fixed(path):
    df = pd.read_csv(path, sep='\s+', header=None, low_memory=False)
    return df

DATA_DIR = '/kaggle/working/project_data'

if not os.path.exists(DATA_DIR):
    print(f"❌ Path doesn't exist {DATA_DIR}")
else:
    print("Loading Data")
    raw_papers  = load_txt_file_fixed(os.path.join(DATA_DIR, "paper_file_ann.txt"))
    raw_authors = load_txt_file_fixed(os.path.join(DATA_DIR, "author_file_ann.txt"))
    raw_train   = load_txt_file_fixed(os.path.join(DATA_DIR, "bipartite_train_ann.txt"))
    raw_test    = load_txt_file_fixed(os.path.join(DATA_DIR, "bipartite_test_ann.txt"))

    raw_papers.columns = ["paper_id", "extra_1"]
    raw_authors.columns = ["author_id", "extra_1"]
    raw_train.columns = ["author_id", "paper_id"]
    raw_test.columns = ["author_id", "paper_id"]

    print("Data Uploaded")

In [ ]:
if not os.path.exists(DATA_DIR):
    print(f"Path doesn't exist: {DATA_DIR}")
else:
    raw_papers  = load_txt_file_fixed(os.path.join(DATA_DIR, "paper_file_ann.txt"))
    raw_authors = load_txt_file_fixed(os.path.join(DATA_DIR, "author_file_ann.txt"))
    raw_train   = load_txt_file_fixed(os.path.join(DATA_DIR, "bipartite_train_ann.txt"))
    raw_test    = load_txt_file_fixed(os.path.join(DATA_DIR, "bipartite_test_ann.txt"))
    features    = load_features(os.path.join(DATA_DIR, "feature.pkl"))

    raw_papers.columns = ["paper_id", "extra_1"]
    raw_authors.columns = ["author_id", "extra_1"]
    raw_train.columns = ["author_id", "paper_id"]
    raw_test.columns = ["author_id", "paper_id"]

    papers_df = raw_papers[['paper_id']].drop_duplicates().reset_index(drop=True)
    authors_df = raw_authors[['author_id']].drop_duplicates().reset_index(drop=True)

    papers_df['paper_id'] = papers_df['paper_id'].astype(int)
    authors_df['author_id'] = authors_df['author_id'].astype(int)

    import matplotlib.pyplot as plt
    import seaborn as sns

    print("\n--- Generating Results ---")
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    top_papers = raw_train['paper_id'].value_counts().head(10)
    sns.barplot(x=top_papers.index, y=top_papers.values, ax=ax[0], palette='Blues_r')
    ax[0].set_title('Top 10 Cited Papers')
    ax[0].set_xlabel('Paper ID')

    top_authors = raw_train['author_id'].value_counts().head(10)
    sns.barplot(x=top_authors.index, y=top_authors.values, ax=ax[1], palette='Greens_r')
    ax[1].set_title('Top 10 Active Authors')
    ax[1].set_xlabel('Author ID')

    plt.show()

    print(f"\nCleaned Papers: {len(papers_df)}")
    print(f"Cleaned Authors: {len(authors_df)}")
    print(f"Training Edges: {len(raw_train)}")

# STEP H- check cleaned data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_cleaning_impact(before_count, after_count, title):
    plt.figure(figsize=(6, 4))
    sns.barplot(x=['Before', 'After'], y=[before_count, after_count], palette='viridis')
    plt.title(f'Data Cleaning Impact: {title}')
    plt.ylabel('Number of Rows')
    plt.show()

print(f"Percentage of valid papers: {(len(papers_df)/len(raw_papers))*100:.2f}%")

In [ ]:
plot_cleaning_impact(len(raw_papers), len(papers_df), "Papers")

plot_cleaning_impact(len(raw_authors), len(authors_df), "Authors")

In [ ]:
import pandas as pd
import os

def load_txt_file_fixed(path):
    return pd.read_csv(path, sep='\s+', header=None, low_memory=False)

DATA_DIR = '/kaggle/working/project_data'

if not os.path.exists(DATA_DIR):
    print(f"Path doens't exist: {DATA_DIR}")
else:
    raw_train = load_txt_file_fixed(os.path.join(DATA_DIR, "bipartite_train_ann.txt"))
    raw_papers = load_txt_file_fixed(os.path.join(DATA_DIR, "paper_file_ann.txt"))

    raw_train.columns = ["author_id", "paper_id"]
    raw_papers.columns = ["paper_id", "extra_1"]

    train_df_cleaned = raw_train.drop_duplicates().reset_index(drop=True)
    papers_df_cleaned = raw_papers.drop_duplicates().reset_index(drop=True)

    summary_data = {
        "Metric": ["Original Row Count", "Cleaned Row Count", "Duplicates Removed"],
        "Value": [len(raw_train), len(train_df_cleaned), len(raw_train) - len(train_df_cleaned)]
    }
    summary_df = pd.DataFrame(summary_data)

    print("Data Cleaning Summary Report:")
    display(summary_df)

    if train_df_cleaned.duplicated().sum() == 0:
        print("\nData is clean and ready.")
        train_df = train_df_cleaned
        papers_df = papers_df_cleaned
        train_df.to_csv("final_train_edges.csv", index=False)
        print("Saved to: final_train_edges.csv")

# **Graph Design**

In [ ]:
import torch
import numpy as np
from torch_geometric.data import HeteroData
import scipy.sparse as sp

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
paper_id_to_idx  = {int(pid): idx for idx, pid in enumerate(papers_df["paper_id"])}
author_id_to_idx = {int(aid): idx for idx, aid in enumerate(authors_df["author_id"])}

NUM_PAPERS  = len(papers_df)
NUM_AUTHORS = len(authors_df)

print(f"Papers  : {NUM_PAPERS}")
print(f"Authors : {NUM_AUTHORS}")

In [ ]:
X = feature_file.toarray() if sp.issparse(feature_file) else np.array(feature_file)
X = X.astype(np.float32)
FEAT_DIM = X.shape[1]

print(f"feature.pkl shape : {X.shape}")
print(f"NUM_AUTHORS       : {NUM_AUTHORS}")
print(f"NUM_PAPERS        : {NUM_PAPERS}")
print(f"Total nodes       : {NUM_AUTHORS + NUM_PAPERS}")

# feature.pkl has 79937 rows — matches NUM_PAPERS only
if X.shape[0] == NUM_PAPERS:
    paper_feats  = torch.tensor(X, dtype=torch.float)
    author_feats = torch.zeros(NUM_AUTHORS, FEAT_DIM)
    print("Features cover papers only. Author features initialised to zeros.")

elif X.shape[0] == NUM_AUTHORS:
    author_feats = torch.tensor(X, dtype=torch.float)
    paper_feats  = torch.zeros(NUM_PAPERS, FEAT_DIM)
    print("Features cover authors only. Paper features initialised to zeros.")

elif X.shape[0] == NUM_AUTHORS + NUM_PAPERS:
    author_feats = torch.tensor(X[:NUM_AUTHORS], dtype=torch.float)
    paper_feats  = torch.tensor(X[NUM_AUTHORS:], dtype=torch.float)
    print("Features cover all nodes.")

else:
    # Partial coverage — pad with zeros for missing nodes
    print(f"Partial coverage ({X.shape[0]} rows). Padding remaining nodes with zeros.")
    if X.shape[0] <= NUM_PAPERS:
        paper_feats       = torch.zeros(NUM_PAPERS,  FEAT_DIM)
        paper_feats[:X.shape[0]] = torch.tensor(X, dtype=torch.float)
        author_feats      = torch.zeros(NUM_AUTHORS, FEAT_DIM)
    else:
        all_feats         = torch.zeros(NUM_AUTHORS + NUM_PAPERS, FEAT_DIM)
        all_feats[:X.shape[0]] = torch.tensor(X, dtype=torch.float)
        author_feats      = all_feats[:NUM_AUTHORS]
        paper_feats       = all_feats[NUM_AUTHORS:]

print(f"author_feats : {author_feats.shape}")
print(f"paper_feats  : {paper_feats.shape}")


In [ ]:
#edge tensor
def build_edges(df, src_col, dst_col, src_map, dst_map):
    src_idx, dst_idx = [], []
    for s, d in zip(df[src_col].astype(int), df[dst_col].astype(int)):
        if s in src_map and d in dst_map and s != d:
            src_idx.append(src_map[s])
            dst_idx.append(dst_map[d])
    return torch.tensor([src_idx, dst_idx], dtype=torch.long)

all_bipartite = pd.concat([raw_train, raw_test]).drop_duplicates()
write_ei  = build_edges(all_bipartite,
                        "author_id", "paper_id",
                        author_id_to_idx, paper_id_to_idx)

cite_df   = raw_papers.rename(columns={"extra_1": "cited_paper_id"})
cite_ei   = build_edges(cite_df,
                        "paper_id", "cited_paper_id",
                        paper_id_to_idx, paper_id_to_idx)

coauth_df = raw_authors.rename(columns={"extra_1": "coauthor_id"})
coauth_ei = build_edges(coauth_df,
                        "author_id", "coauthor_id",
                        author_id_to_idx, author_id_to_idx)

print(f"writes   edges : {write_ei.shape[1]}")
print(f"cites    edges : {cite_ei.shape[1]}")
print(f"coauthor edges : {coauth_ei.shape[1]}")

In [ ]:
# ── PART 4: Build HeteroData — Nodes + Forward + Reverse Edges ────────
data = HeteroData()

data['author'].x       = author_feats
data['paper'].x        = paper_feats
data['author'].node_id = torch.tensor(authors_df["author_id"].values, dtype=torch.long)
data['paper'].node_id  = torch.tensor(papers_df["paper_id"].values,   dtype=torch.long)

data['author', 'writes',     'paper' ].edge_index = write_ei
data['paper',  'cites',      'paper' ].edge_index = cite_ei
data['author', 'coauthor',   'author'].edge_index = coauth_ei

data['paper',  'written_by', 'author'].edge_index = write_ei.flip(0)
data['paper',  'cited_by',   'paper' ].edge_index = cite_ei.flip(0)
data['author', 'co_of',      'author'].edge_index = coauth_ei.flip(0)

print("Node types :", data.node_types)
print("Edge types :", data.edge_types)

In [ ]:
#edge weight
# writes / written_by — symmetric, same weight both directions
write_w = torch.ones(write_ei.shape[1])
data['author', 'writes',     'paper' ].edge_weight = write_w
data['paper',  'written_by', 'author'].edge_weight = write_w

# cites — normalised by out-degree of citing paper (how many it cites)
cite_src = cite_ei[0]
cite_dst = cite_ei[1]

out_deg = torch.zeros(NUM_PAPERS)
out_deg.scatter_add_(0, cite_src, torch.ones(cite_src.shape[0]))
out_deg = out_deg.clamp(min=1)
cites_w = 1.0 / out_deg[cite_src]

# cited_by — normalised by in-degree of cited paper (how many times cited)
in_deg = torch.zeros(NUM_PAPERS)
in_deg.scatter_add_(0, cite_dst, torch.ones(cite_dst.shape[0]))
in_deg = in_deg.clamp(min=1)
cited_by_w = 1.0 / in_deg[cite_dst]

data['paper', 'cites',    'paper'].edge_weight = cites_w
data['paper', 'cited_by', 'paper'].edge_weight = cited_by_w

# coauthor / co_of — symmetric, same weight both directions
coauth_w = torch.ones(coauth_ei.shape[1])
data['author', 'coauthor', 'author'].edge_weight = coauth_w
data['author', 'co_of',    'author'].edge_weight = coauth_w

print(f"writes    weight : {write_w.min().item():.3f} – {write_w.max().item():.3f}")
print(f"cites     weight : {cites_w.min().item():.3f} – {cites_w.max().item():.3f}")
print(f"cited_by  weight : {cited_by_w.min().item():.3f} – {cited_by_w.max().item():.3f}")
print(f"coauthor  weight : {coauth_w.min().item():.3f} – {coauth_w.max().item():.3f}")


In [ ]:
# ── PART 6: Train / Test Masks, Validation & Save ─────────────────────

train_pairs = {
    (author_id_to_idx[int(r.author_id)], paper_id_to_idx[int(r.paper_id)])
    for _, r in raw_train.iterrows()
    if int(r.author_id) in author_id_to_idx and int(r.paper_id) in paper_id_to_idx
}
test_pairs = {
    (author_id_to_idx[int(r.author_id)], paper_id_to_idx[int(r.paper_id)])
    for _, r in raw_test.iterrows()
    if int(r.author_id) in author_id_to_idx and int(r.paper_id) in paper_id_to_idx
}

ei = data['author', 'writes', 'paper'].edge_index
data['author', 'writes', 'paper'].train_mask = torch.tensor(
    [(int(ei[0, i]), int(ei[1, i])) in train_pairs for i in range(ei.shape[1])],
    dtype=torch.bool
)
data['author', 'writes', 'paper'].test_mask = torch.tensor(
    [(int(ei[0, i]), int(ei[1, i])) in test_pairs for i in range(ei.shape[1])],
    dtype=torch.bool
)

print(f"train_mask : {data['author', 'writes', 'paper'].train_mask.sum().item()} edges")
print(f"test_mask  : {data['author', 'writes', 'paper'].test_mask.sum().item()} edges")

assert len(data.edge_types) == 6
assert data['author'].x.shape[0] == NUM_AUTHORS
assert data['paper'].x.shape[0]  == NUM_PAPERS
print(data)
print("✓ Graph ready for GNN training")

torch.save(data, "hetero_graph.pt")
print("✓ Saved → hetero_graph.pt")

In [ ]:
from google.colab import files

In [ ]:
# files.download("hetero_graph.pt")

# **MODELing**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#    **** Encoder  (PyG · SAGEConv · edge weights · layer fusion) ****
# ══════════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv


class FeatureFusion(nn.Module):
    def __init__(self, feat_dims: dict, out_dim: int):
        super().__init__()
        self.projections = nn.ModuleDict({
            ntype: nn.Sequential(
                nn.Linear(dim, out_dim),
                nn.LayerNorm(out_dim),
                nn.GELU(),
            )
            for ntype, dim in feat_dims.items()
        })

    def forward(self, x_dict: dict) -> dict:
        return {ntype: self.projections[ntype](x) for ntype, x in x_dict.items()}

In [ ]:
class Encoder(nn.Module):

    def __init__(
        self,
        in_feats: int,
        hid_feats: int,
        out_feats: int,          # kept for API compatibility; fusion sets final dim
        edge_names: list,        # list of canonical edge tuples (src, rel, dst)
        n_layers: int = 3,
        activation=F.gelu,
        dropout: float = 0.3,
        aggregate: str = 'mean',
    ):
        super().__init__()
        self.activation = activation
        self.dropout    = dropout
        self.n_layers   = n_layers

        dims = [in_feats] + [hid_feats] * n_layers
        self.convs = nn.ModuleList()
        for i in range(n_layers):
            conv = HeteroConv(
                {
                    etype: SAGEConv(
                        (dims[i], dims[i]),
                        dims[i + 1],
                        aggr=aggregate,
                        normalize=True,
                        bias=True,
                    )
                    for etype in edge_names
                },
                aggr=aggregate,
            )
            self.convs.append(conv)

        self.layer_norms = nn.ModuleDict()
        self._hid = hid_feats

        self._res_proj = nn.ModuleDict()   # built lazily

    # ------------------------------------------------------------------
    def _get_norm(self, ntype: str, layer_idx: int, dim: int) -> nn.LayerNorm:
        key = f"{ntype}_{layer_idx}"
        if key not in self.layer_norms:
            self.layer_norms[key] = nn.LayerNorm(dim).to(
                next(self.parameters()).device
            )
        return self.layer_norms[key]

    def _get_res_proj(self, ntype: str, in_dim: int, out_dim: int) -> nn.Linear:
        if ntype not in self._res_proj:
            self._res_proj[ntype] = nn.Linear(in_dim, out_dim, bias=False).to(
                next(self.parameters()).device
            )
        return self._res_proj[ntype]

    # ------------------------------------------------------------------
    def forward(self, x_dict: dict, edge_index_dict: dict,
                edge_weight_dict: dict = None) -> dict:
        layer_outputs = []
        h = x_dict

        for l, conv in enumerate(self.convs):
            edge_attr_kw = {}
            if edge_weight_dict is not None:

                edge_attr_kw = {
                    etype: {'edge_weight': edge_weight_dict[etype]}
                    for etype in edge_index_dict
                    if etype in edge_weight_dict
                }

            h_new = conv(h, edge_index_dict,
                         **({'edge_weight_dict': edge_attr_kw}
                            if edge_attr_kw else {}))

            h_res = {}
            for ntype in h_new:
                out  = h_new[ntype]
                prev = h.get(ntype)

                if prev is not None and prev.shape[-1] != out.shape[-1]:
                    prev = self._get_res_proj(ntype, prev.shape[-1], out.shape[-1])(prev)

                out = self._get_norm(ntype, l, out.shape[-1])(
                    out + prev if prev is not None else out
                )

                if l < self.n_layers - 1:
                    out = self.activation(out)
                    out = F.dropout(out, self.dropout, self.training)

                h_res[ntype] = out

            layer_outputs.append(h_res)
            h = h_res

        # ── Multi-layer embedding fusion ──────────────────────────────
        all_ntypes = set(nt for lo in layer_outputs for nt in lo)
        fused = {}
        for ntype in all_ntypes:
            layer_vecs = [
                lo[ntype] for lo in layer_outputs if ntype in lo
            ]
            mean_vec  = torch.stack(layer_vecs, dim=0).mean(dim=0)
            fused[ntype] = torch.cat(layer_vecs + [mean_vec], dim=-1)

        return fused

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                     ****** Predictors ******
# ═══════════════════════════════════════════════════════════════════

class ScorePredictor(nn.Module):
    """Dot-product baseline predictor (unchanged API, works on fused embeddings)."""

    def forward(self, edge_index: torch.Tensor,
                x_src: torch.Tensor, x_dst: torch.Tensor) -> torch.Tensor:
        src_idx, dst_idx = edge_index
        return (x_src[src_idx] * x_dst[dst_idx]).sum(dim=-1, keepdim=True)

    def get_score(self, h_u: torch.Tensor, h_v: torch.Tensor) -> torch.Tensor:
        return (h_u * h_v).sum(dim=-1, keepdim=True)


class MLPPredictor(nn.Module):
    """
    Advanced link-prediction head with:
      • Deep residual MLP
      • Hybrid score = MLP(concat) + cosine similarity
      • BatchNorm between layers for training stability
    """

    def __init__(self, in_features: int, out_classes: int = 1):
        super().__init__()
        h1 = in_features
        h2 = max(in_features // 2, 64)
        h3 = max(in_features // 4, 32)
        h4 = max(in_features // 8, 16)

        # ── Residual MLP trunk ────────────────────────────────────────
        self.fc1   = nn.Linear(2 * in_features, h1)
        self.bn1   = nn.BatchNorm1d(h1)
        self.fc2   = nn.Linear(h1, h2)
        self.bn2   = nn.BatchNorm1d(h2)
        self.fc3   = nn.Linear(h2, h3)
        self.bn3   = nn.BatchNorm1d(h3)
        self.fc4   = nn.Linear(h3, h4)
        self.bn4   = nn.BatchNorm1d(h4)
        self.fc_out = nn.Linear(h4, out_classes)

        # Skip connections to keep gradient flow through depth
        self.skip1 = nn.Linear(2 * in_features, h2)  # input  → after block1
        self.skip2 = nn.Linear(h2, h4)               # h2     → after block2

        # Jump connection: direct input → output (wide residual)
        self.jump  = nn.Linear(2 * in_features, out_classes)

        self.drop  = nn.Dropout(0.2)

    # ------------------------------------------------------------------
    def get_score(self, h_u: torch.Tensor, h_v: torch.Tensor) -> torch.Tensor:
        """
        Hybrid score = MLP(concat[h_u, h_v])  +  cosine_similarity(h_u, h_v)
        """
        pair = torch.cat([h_u, h_v], dim=-1)               # [B, 2F]

        # Block 1
        x = self.drop(F.gelu(self.bn1(self.fc1(pair))))
        x = self.drop(F.gelu(self.bn2(self.fc2(x))))
        x = x + self.skip1(pair)                            # residual skip

        # Block 2
        skip_x = x
        x = self.drop(F.gelu(self.bn3(self.fc3(x))))
        x = self.drop(F.gelu(self.bn4(self.fc4(x))))
        x = x + self.skip2(skip_x)                         # residual skip

        mlp_score  = self.fc_out(x) + self.jump(pair)      # wide residual

        # Cosine similarity signal (normalise to [-1,1] then scale)
        cos_score = F.cosine_similarity(h_u, h_v, dim=-1).unsqueeze(-1)

        return mlp_score + cos_score

    # ------------------------------------------------------------------
    def forward(self, edge_index: torch.Tensor,
                x_src: torch.Tensor, x_dst: torch.Tensor) -> torch.Tensor:
        """
        Args:
            edge_index : LongTensor [2, E]
            x_src      : Tensor [N_src, F]
            x_dst      : Tensor [N_dst, F]
        Returns:
            scores     : Tensor [E, 1]
        """
        src_idx, dst_idx = edge_index
        return self.get_score(x_src[src_idx], x_dst[dst_idx])

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#            ****** Model  (end-to-end wrapper) ******
# ═══════════════════════════════════════════════════════════════════

class Model(nn.Module):
    """
    Full heterogeneous GNN for link prediction.

    Pipeline
    ────────
    1.  FeatureFusion   — project each node type to a common dim
    2.  Encoder         — SAGEConv layers with edge weights, residuals, LayerNorm
    3.  MLPPredictor /  — score positive & negative edge pairs
        ScorePredictor
    """

    def __init__(
        self,
        feat_dims: dict,
        in_features: int,
        hidden_features: int,
        out_features: int,
        edge_names: list,
        n_layers: int = 3,
        activation=F.gelu,
        dropout: float = 0.3,
        pred_opt: str = 'mlp',
    ):
        super().__init__()
        self.pred_opt = pred_opt

        # 1. Feature fusion (projects all node types to in_features)
        self.feature_fusion = FeatureFusion(feat_dims, in_features)

        # 2. Encoder (fused output dim = hid * (n_layers + 1))
        self.encoder = Encoder(
            in_feats=in_features,
            hid_feats=hidden_features,
            out_feats=out_features,
            edge_names=edge_names,
            n_layers=n_layers,
            activation=activation,
            dropout=dropout,
        )

        # Fused embedding dim = hidden_features * (n_layers + 1)
        fused_dim = hidden_features * (n_layers + 1)

        # 3. Predictors
        self.pred_mlp  = MLPPredictor(fused_dim, out_classes=1)
        self.pred_base = ScorePredictor()

    # ------------------------------------------------------------------
    def encode(self, x_dict: dict, edge_index_dict: dict,
               edge_weight_dict: dict = None) -> dict:
        h = self.feature_fusion(x_dict)
        h = self.encoder(h, edge_index_dict, edge_weight_dict)
        return h

    # ------------------------------------------------------------------
    def forward(
        self,
        x_dict:           dict,
        edge_index_dict:  dict,
        pos_edge_index:   torch.Tensor,   # [2, E_pos]  for target relation
        neg_edge_index:   torch.Tensor,   # [2, E_neg]
        src_type:         str,
        dst_type:         str,
        edge_weight_dict: dict = None,
    ):
        """
        Returns:
            pos_score : Tensor [E_pos, 1]
            neg_score : Tensor [E_neg, 1]
        """
        torch.cuda.empty_cache()
        h = self.encode(x_dict, edge_index_dict, edge_weight_dict)

        x_src = h[src_type]
        x_dst = h[dst_type]

        if self.pred_opt == 'base':
            pos_score = self.pred_base(pos_edge_index, x_src, x_dst)
            neg_score = self.pred_base(neg_edge_index, x_src, x_dst)
        else:
            pos_score = self.pred_mlp(pos_edge_index, x_src, x_dst)
            neg_score = self.pred_mlp(neg_edge_index, x_src, x_dst)

        return pos_score, neg_score

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#            ****** Model  (end-to-end wrapper) ******
# ═══════════════════════════════════════════════════════════════════

class Model(nn.Module):
    """
    Full heterogeneous GNN for link prediction.

    Pipeline
    ────────
    1.  FeatureFusion   — project each node type to a common dim
    2.  Encoder         — SAGEConv layers with edge weights, residuals, LayerNorm
    3.  MLPPredictor /  — score positive & negative edge pairs
        ScorePredictor
    """

    def __init__(
        self,
        feat_dims: dict,
        in_features: int,
        hidden_features: int,
        out_features: int,
        edge_names: list,
        n_layers: int = 3,
        activation=F.gelu,
        dropout: float = 0.3,
        pred_opt: str = 'mlp',
    ):
        super().__init__()
        self.pred_opt = pred_opt

        # 1. Feature fusion (projects all node types to in_features)
        self.feature_fusion = FeatureFusion(feat_dims, in_features)

        # 2. Encoder (fused output dim = hid * (n_layers + 1))
        self.encoder = Encoder(
            in_feats=in_features,
            hid_feats=hidden_features,
            out_feats=out_features,
            edge_names=edge_names,
            n_layers=n_layers,
            activation=activation,
            dropout=dropout,
        )

        # Fused embedding dim = hidden_features * (n_layers + 1)
        fused_dim = hidden_features * (n_layers + 1)

        # 3. Predictors
        self.pred_mlp  = MLPPredictor(fused_dim, out_classes=1)
        self.pred_base = ScorePredictor()

    # ------------------------------------------------------------------
    def encode(self, x_dict: dict, edge_index_dict: dict,
               edge_weight_dict: dict = None) -> dict:
        h = self.feature_fusion(x_dict)
        h = self.encoder(h, edge_index_dict, edge_weight_dict)
        return h

    # ------------------------------------------------------------------
    def forward(
        self,
        x_dict:           dict,
        edge_index_dict:  dict,
        pos_edge_index:   torch.Tensor,   # [2, E_pos]  for target relation
        neg_edge_index:   torch.Tensor,   # [2, E_neg]
        src_type:         str,
        dst_type:         str,
        edge_weight_dict: dict = None,
    ):
        """
        Returns:
            pos_score : Tensor [E_pos, 1]
            neg_score : Tensor [E_neg, 1]
        """
        torch.cuda.empty_cache()
        h = self.encode(x_dict, edge_index_dict, edge_weight_dict)

        x_src = h[src_type]
        x_dst = h[dst_type]

        if self.pred_opt == 'base':
            pos_score = self.pred_base(pos_edge_index, x_src, x_dst)
            neg_score = self.pred_base(neg_edge_index, x_src, x_dst)
        else:
            pos_score = self.pred_mlp(pos_edge_index, x_src, x_dst)
            neg_score = self.pred_mlp(neg_edge_index, x_src, x_dst)

        return pos_score, neg_score

> ###  T RandomLinkSplit

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
import torch_geometric.transforms as T

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=1.0,
    add_negative_train_samples=True,
    edge_types=[('author', 'writes', 'paper')],
    rev_edge_types=[('paper', 'written_by', 'author')] # Ensure this matches your reverse edge name
)
train_data, val_data, test_data = transform(data)


graphsage_model = Model(
    feat_dims={'author': FEAT_DIM, 'paper': FEAT_DIM},
    in_features=128,
    hidden_features=64,
    out_features=64,
    edge_names=train_data.edge_types,
    pred_opt='mlp'
)

optimizer_sage = optim.Adam(graphsage_model.parameters(), lr=0.005)
criterion_sage = torch.nn.BCEWithLogitsLoss()



def train_graphsage():
    graphsage_model.train()
    optimizer_sage.zero_grad()

    target_edge = ('author', 'writes', 'paper')

    pred_edges = train_data[target_edge].edge_label_index
    labels = train_data[target_edge].edge_label

    pos_edges = pred_edges[:, labels == 1]
    neg_edges = pred_edges[:, labels == 0]

    pos_score, neg_score = graphsage_model(
        x_dict=train_data.x_dict,
        edge_index_dict=train_data.edge_index_dict,
        pos_edge_index=pos_edges,
        neg_edge_index=neg_edges,
        src_type='author',
        dst_type='paper'
    )

    preds = torch.cat([pos_score.squeeze(), neg_score.squeeze()])
    true_labels = torch.cat([
        torch.ones(pos_score.size(0)),
        torch.zeros(neg_score.size(0))
    ]).to(preds.device)

    loss = criterion_sage(preds, true_labels)
    loss.backward()
    optimizer_sage.step()

    return loss.item()


def evaluate_graphsage(data_split):
    graphsage_model.eval()
    with torch.no_grad():
        target_edge = ('author', 'writes', 'paper')

        pred_edges = data_split[target_edge].edge_label_index
        labels = data_split[target_edge].edge_label

        pos_edges = pred_edges[:, labels == 1]
        neg_edges = pred_edges[:, labels == 0]

        pos_score, neg_score = graphsage_model(
            x_dict=data_split.x_dict,
            edge_index_dict=data_split.edge_index_dict,
            pos_edge_index=pos_edges,
            neg_edge_index=neg_edges,
            src_type='author',
            dst_type='paper'
        )

        preds = torch.cat([pos_score.squeeze(), neg_score.squeeze()]).cpu().numpy()
        true_labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).cpu().numpy()

        return roc_auc_score(true_labels, preds)

print("Starting GraphSAGE Training...")
for epoch in range(1, 51):
    loss = train_graphsage()

    if epoch % 5 == 0:
        val_auc = evaluate_graphsage(val_data)
        print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Validation AUC: {val_auc:.4f}")

test_auc = evaluate_graphsage(test_data)
print(f" Final Test AUC: {test_auc:.4f}")

torch.save(graphsage_model.state_dict(), "graphsage_weights.pth")
print("GraphSAGE model weights saved to 'graphsage_weights.pth'")

In [ ]:
torch.save(graphsage_model.state_dict(), "graphsage_weights.pth")

> ###  LightGCN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import LightGCN
from sklearn.metrics import roc_auc_score


class LightGCNRecommender(nn.Module):
    """
    State-of-the-art LightGCN architecture optimized for Author-Paper Recommendation.
    """
    def __init__(
        self,
        num_authors: int,
        num_papers: int,
        embedding_dim: int = 64,
        num_layers: int = 3
    ):
        super().__init__()

        # 1. Initialize learnable node embeddings
        self.num_authors = num_authors
        self.num_papers = num_papers
        self.embedding_dim = embedding_dim

        # Combine authors and papers into one large embedding table
        self.embedding = nn.Embedding(num_authors + num_papers, embedding_dim)

        # 2. The LightGCN Message Passing Engine
        self.convs = LightGCN(
            num_nodes=num_authors + num_papers,
            embedding_dim=embedding_dim,
            num_layers=num_layers
        )

    def forward(self, edge_index_dict, pos_edge_index, neg_edge_index):
        # --- Step 1: Homogeneous Graph Conversion ---
        writes_edge = edge_index_dict[('author', 'writes', 'paper')]
        shifted_papers = writes_edge[1] + self.num_authors
        homo_edge_index = torch.stack([writes_edge[0], shifted_papers])

        homo_edge_index = homo_edge_index.long()
        # --- Step 2: Message Passing ---
        all_embeddings = self.convs.get_embedding(homo_edge_index)

        # Split them back into authors and papers
        author_emb = all_embeddings[:self.num_authors]
        paper_emb  = all_embeddings[self.num_authors:]

        # --- Step 3: Link Prediction (Dot Product) ---
        pos_src_emb = author_emb[pos_edge_index[0]]
        pos_dst_emb = paper_emb[pos_edge_index[1]]
        pos_score = (pos_src_emb * pos_dst_emb).sum(dim=-1)

        neg_src_emb = author_emb[neg_edge_index[0]]
        neg_dst_emb = paper_emb[neg_edge_index[1]]
        neg_score = (neg_src_emb * neg_dst_emb).sum(dim=-1)

        return pos_score, neg_score

lightgcn_model = LightGCNRecommender(
    num_authors=NUM_AUTHORS,
    num_papers=NUM_PAPERS,
    embedding_dim=64,
    num_layers=3
)

optimizer_lightgcn = optim.Adam(lightgcn_model.parameters(), lr=0.005)
criterion_lightgcn = torch.nn.BCEWithLogitsLoss()


def train_lightgcn():
    lightgcn_model.train()
    optimizer_lightgcn.zero_grad()

    target_edge = ('author', 'writes', 'paper')

    pred_edges = train_data[target_edge].edge_label_index
    labels = train_data[target_edge].edge_label

    pos_edges = pred_edges[:, labels == 1]
    neg_edges = pred_edges[:, labels == 0]

    # Notice we don't need to pass x_dict or src_type here for LightGCN!
    pos_score, neg_score = lightgcn_model(
        edge_index_dict=train_data.edge_index_dict,
        pos_edge_index=pos_edges,
        neg_edge_index=neg_edges
    )

    # Combine scores and labels for the loss function
    preds = torch.cat([pos_score.squeeze(), neg_score.squeeze()])
    true_labels = torch.cat([
        torch.ones(pos_score.size(0)),
        torch.zeros(neg_score.size(0))
    ]).to(preds.device)

    loss = criterion_lightgcn(preds, true_labels)
    loss.backward()
    optimizer_lightgcn.step()

    return loss.item()


def evaluate_lightgcn(data_split):
    lightgcn_model.eval()
    with torch.no_grad():
        target_edge = ('author', 'writes', 'paper')
        pred_edges = data_split[target_edge].edge_label_index
        labels = data_split[target_edge].edge_label

        pos_edges = pred_edges[:, labels == 1]
        neg_edges = pred_edges[:, labels == 0]

        pos_score, neg_score = lightgcn_model(
            edge_index_dict=data_split.edge_index_dict,
            pos_edge_index=pos_edges,
            neg_edge_index=neg_edges
        )

        preds = torch.cat([pos_score.squeeze(), neg_score.squeeze()]).cpu().numpy()
        true_labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).cpu().numpy()

        return roc_auc_score(true_labels, preds)

# ==========================================
# 5. Execute Training & Save Weights
# ==========================================
print("Starting LightGCN Training...")
for epoch in range(1, 51):
    loss = train_lightgcn()

    # Print metrics every 5 epochs
    if epoch % 5 == 0:
        val_auc = evaluate_lightgcn(val_data)
        print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Validation AUC: {val_auc:.4f}")

# Final Test Score
test_auc = evaluate_lightgcn(test_data)
print(f"Final Test AUC: {test_auc:.4f}")

# Save the model for Streamlit
torch.save(lightgcn_model.state_dict(), "lightgcn_weights.pth")
print("LightGCN model weights saved to 'lightgcn_weights.pth'")

In [ ]:
torch.save(lightgcn_model.state_dict(), "lightgcn_weights.pth")

### **UI using ngrok-streamlit**

In [ ]:
!pip install streamlit torch_geometric pyngrok -q


In [ ]:
import os
import shutil
import subprocess
import time
from pyngrok import ngrok


SRC_APP = '/kaggle/input/datasets/mohamedashraf2005/new-applied-ui-model/app.py'
SRC_GRAPH = '/kaggle/input/notebooks/anasmelgezawy/notebook6aca299583/hetero_graph.pt'
SRC_SAGE = '/kaggle/input/notebooks/anasmelgezawy/notebook6aca299583/graphsage_weights.pth'
SRC_GCN = '/kaggle/input/notebooks/anasmelgezawy/notebook6aca299583/lightgcn_weights.pth'

WORKING_DIR = '/kaggle/working'
os.chdir(WORKING_DIR)

os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
os.system("fuser -k 8505/tcp")
time.sleep(2)
print("Ports forcefully cleared and isolated.")

shutil.copy(SRC_APP, 'app.py')
shutil.copy(SRC_GRAPH, 'hetero_graph.pt')
shutil.copy(SRC_SAGE, 'graphsage_weights.pth')
shutil.copy(SRC_GCN, 'lightgcn_weights.pth')

print("\n Verifying file copies and file sizes:")
for f in ['app.py', 'hetero_graph.pt', 'graphsage_weights.pth', 'lightgcn_weights.pth']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f" -> {f} exists ({size_mb:.2f} MB)")
    else:
        print(f" ->  ERROR: {f} failed to copy!")

with open('app.py', 'r') as file:
    code = file.read()

code = code.replace('@st.cache_resource', '# Caching removed for emergency stability')

with open('app.py', 'w') as file:
    file.write(code)

print("\n Launching Streamlit pipeline...")
with open("streamlit_output.log", "w") as log_file:
    proc = subprocess.Popen([
        "streamlit", "run", "app.py", 
        "--server.port=8505", 
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ], stdout=log_file, stderr=log_file)

print(" Initializing graph tensors (Waiting 12 seconds)...")
time.sleep(12)

# 6. Bind the proxy tunnel
if proc.poll() is not None:
    print("\n Streamlit failed to start. Printing crash trace:")
    with open("streamlit_output.log", "r") as f:
        print(f.read())
else:
    print("Streamlit server status: ONLINE")
    ngrok.set_auth_token()
    try:
        ngrok.disconnect(public_url=None)
    except:
        pass
        
    public_url = ngrok.connect("127.0.0.1:8505")
    
    print("\n" + "="*65)
    print("SUBMISSION LINK IS LIVE:")
    print(f"{public_url} ")
  